# 05 — Held-out WBC localization evaluation

**Purpose:** Evaluate the completed best.pt on the test split only, with official Ultralytics detection metrics.

**Required inputs:** Completed main run from notebook 04 and unchanged processed dataset.

**Saved outputs:** `outputs/paper_results/detection_metrics.csv/.json` and `outputs/figures/test_evaluation/<run>/`.

## 1. Load the central configuration

The root is found from the notebook location or the installed package. Set `LEUKEMIA_YOLO_CONFIG` only for synthetic smoke tests or an alternate config.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "config.py").is_file():
        sys.path.insert(0, str(candidate))
        break
from src.config import load_config, ensure_outputs
cfg = load_config()
out = ensure_outputs(cfg)
os.chdir(cfg["root"])
print("Project root:", cfg["root"])
print("Configuration:", cfg["config_path"])

## 2. Resolve exact checkpoint and test size

Checkpoint provenance is checked against the frozen dataset fingerprint.

In [ ]:
from src.training import resolve_checkpoint
from src.validation import validate_processed
checkpoint = resolve_checkpoint(cfg)
data_check = validate_processed(cfg["processed_data_path"])
print("Checkpoint:", checkpoint)
print("Held-out test images:", data_check["splits"]["test"]["images"])

## 3. Run test evaluation

The shared evaluator invokes `YOLO(best.pt).val(data=..., split='test', plots=True)` and reads box precision/recall/mAP.

In [ ]:
from src.evaluation import evaluate, evaluation_figure_paths
evaluation = evaluate(cfg)
import pandas as pd
display(pd.DataFrame([evaluation["metrics"]]))
print("Evaluation figures:", evaluation["figures"])

## 4. Show curves and confusion matrix

Framework plots are generated from the held-out split. Missing optional plot files are labeled explicitly.

In [ ]:
from IPython.display import display, Image as IPImage
for name,path in evaluation_figure_paths(evaluation).items():
    print(name, path if path.is_file() else "Not available")
    if path.is_file(): display(IPImage(filename=str(path)))

## 5. Saved result and interpretation

Metrics measure WBC localization. They do not measure leukemia or blast classification.

In [ ]:
assert evaluation["split"] == "test"
assert (out / "paper_results" / "detection_metrics.json").is_file()
print("PASS — test-only WBC localization results saved for this checkpoint. Next: notebook 06.")

## Next notebook

Open **06_WBC_Detection.ipynb**.